# Feature Engineering — Red SCADA
Eliminar features inútiles y crear nuevas basadas en el EDA

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

DELTA_FINAL_PATH = "/Volumes/workspace/default/network_data/features_delta/"
DELTA_FE_PATH    = "/Volumes/workspace/default/network_data/features_fe/"

df = spark.read.format("delta").load(DELTA_FINAL_PATH)
print(f"Columnas originales : {len(df.columns)}")
print(f"Total ventanas      : {df.count():,}")

## 1 — Eliminar features sin señal

In [0]:
# Features a eliminar según EDA:
# - Correlación con label < 0.005 (ruido puro)
# - Varianza cero (constantes)
# - 100% un solo valor (udp_packets siempre 0)

drop_cols = [
    # Protocolos de escritorio/red — no discriminan en entorno SCADA
    "has_vnc",
    "has_remote_desktop_protocol",
    "has_web_browsing",
    "has_server_message_block_smb",
    "has_kaspersky_lab_update",
    "has_google_chrome",
    "has_ssdp",
    "has_simple_object_access_protocol",
    "has_corrupt_proto",
    "has_corrupt_function_code",
    "has_corrupt_dst_port",
    # Volumen — idéntico en Normal y Ataque
    "udp_packets",           # siempre 0
    "inbound_packets",       # sin diferencia entre clases
    # Duración mínima — 0 en ambas clases
    "min_connection_duration_ms",
    # Transaction ID — correlación < 0.01
    "transaction_id_mean",
    # Gaps — constantes (sin varianza)
    "gap_before_seconds",
    "is_after_gap",
]

# Filtrar solo los que existen en el df
drop_cols = [c for c in drop_cols if c in df.columns]
print(f"Eliminando {len(drop_cols)} features: {drop_cols}")

df_fe = df.drop(*drop_cols)
print(f"Columnas restantes  : {len(df_fe.columns)}")

## 2 — Crear nuevas features

In [0]:
# ── 1. duration_ratio
# avg / max — captura la forma de la distribución de duraciones
# Si es cercano a 1: todas las conexiones duran igual (patrón ataque)
# Si es bajo: hay conexiones muy largas mezcladas con cortas (patrón normal)
df_fe = df_fe.withColumn(
    "duration_ratio",
    F.when(
        F.col("max_connection_duration_ms") > 0,
        F.col("avg_connection_duration_ms") / F.col("max_connection_duration_ms")
    ).otherwise(None)
)

# ── 2. duration_cv (coeficiente de variación)
# std / mean de duración — ataques tienen cv muy bajo (conexiones uniformes)
# Necesitamos std: la calculamos desde avg y max como proxy
std_col = "std_connection_duration_ms" if "std_connection_duration_ms" in df_fe.columns else None

if std_col:
    df_fe = df_fe.withColumn(
        "duration_cv",
        F.when(
            F.col("avg_connection_duration_ms") > 0,
            F.col(std_col) / F.col("avg_connection_duration_ms")
        ).otherwise(None)
    )
    print("duration_cv creado desde std_connection_duration_ms")
else:
    # Proxy: (max - avg) / avg como indicador de dispersión
    df_fe = df_fe.withColumn(
        "duration_spread",
        F.when(
            F.col("avg_connection_duration_ms") > 0,
            (F.col("max_connection_duration_ms") - F.col("avg_connection_duration_ms"))
            / F.col("avg_connection_duration_ms")
        ).otherwise(None)
    )
    print("duration_cv no disponible — creado duration_spread como proxy")

# ── 3. is_short_duration (flag binario)
# Umbral basado en el EDA: Normal ~300M ms, Ataque ~143M ms
# Punto de corte natural en torno a 200M ms
UMBRAL_DURATION = 200_000_000

df_fe = df_fe.withColumn(
    "is_short_duration",
    F.when(F.col("avg_connection_duration_ms") < UMBRAL_DURATION, 1).otherwise(0)
)

# ── 4. packets_per_ms
# Densidad de paquetes por ms de conexión
# Combina volumen con duración — puede capturar ataques de inyección rápida
df_fe = df_fe.withColumn(
    "packets_per_ms",
    F.when(
        F.col("avg_connection_duration_ms") > 0,
        F.col("total_packets") / (F.col("avg_connection_duration_ms") / 1_000_000.0)
    ).otherwise(None)
)

nuevas = ["duration_ratio", "is_short_duration", "packets_per_ms"]
if std_col:
    nuevas.append("duration_cv")
else:
    nuevas.append("duration_spread")

print(f"\nFeatures nuevas creadas: {nuevas}")

## 3 — Verificar las nuevas features

In [0]:
# Estadísticas de las nuevas features por label
nuevas_check = [c for c in ["duration_ratio", "duration_cv", "duration_spread",
                              "is_short_duration", "packets_per_ms"] if c in df_fe.columns]

display(
    df_fe.groupBy("label")
         .agg(*[F.round(F.mean(c), 6).alias(f"mean_{c}") for c in nuevas_check])
         .orderBy("label")
)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Distribución de duration_ratio por clase
pdf_new = df_fe.select(nuevas_check + ["label"]).sample(fraction=0.2, seed=42).toPandas()

n_plots = len(nuevas_check)
fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4))
if n_plots == 1:
    axes = [axes]

colors = {"Normal": "#4C8BF5", "Ataque": "#E8453C"}

for ax, col in zip(axes, nuevas_check):
    normal = pdf_new[pdf_new["label"] == 0][col].dropna()
    ataque = pdf_new[pdf_new["label"] == 1][col].dropna()
    ax.hist(normal, bins=50, color="#4C8BF5", alpha=0.7, label="Normal (0)", density=True)
    ax.hist(ataque, bins=50, color="#E8453C", alpha=0.7, label="Ataque (1)", density=True)
    ax.set_title(col, fontsize=11, fontweight="bold")
    ax.set_xlabel("Valor")
    ax.set_ylabel("Densidad")
    ax.legend(fontsize=8)

plt.suptitle("Distribución de nuevas features — Normal vs Ataque", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 4 — Correlación de nuevas features con label

In [0]:
all_num = [c for c in df_fe.columns
           if df_fe.schema[c].dataType.simpleString() in ("double", "long", "int", "float")
           and c not in ("label", "window_id", "session_id")]

pdf_corr = df_fe.select(all_num + ["label"]).sample(fraction=0.1, seed=42).toPandas()

corr_label = (
    pdf_corr[all_num + ["label"]]
    .corr()["label"]
    .drop("label")
    .abs()
    .sort_values(ascending=False)
    .reset_index()
)
corr_label.columns = ["feature", "corr_abs"]

print("Ranking completo de correlación con label:")
print(corr_label.to_string(index=False))

top = corr_label.head(15)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top["feature"][::-1], top["corr_abs"][::-1], color="#4C8BF5", edgecolor="white")
ax.axvline(0.1, color="gray", linestyle="--", linewidth=0.8, label="umbral 0.1")
ax.set_title("Top 15 features — correlación absoluta con label (post FE)", fontsize=12, fontweight="bold")
ax.set_xlabel("Correlación absoluta")
ax.legend()
plt.tight_layout()
plt.show()

## 5 — Guardar dataset final con feature engineering

In [0]:
print(f"Columnas finales   : {len(df_fe.columns)}")
print(f"Columnas originales: {len(df.columns)}")
print(f"Reducción          : {len(df.columns) - len(df_fe.columns)} columnas eliminadas/transformadas")
print(f"Nuevas añadidas    : {len(nuevas_check)}")
print()
print("Schema final:")
df_fe.printSchema()

In [0]:
# Ratio write/read — en ataques de inyección aumentan las escrituras
# ya tienes write_operations y read_requests
df_fe = df_fe.withColumn(
    "write_read_ratio_safe",
    F.when(
        F.col("read_requests") > 0,
        F.col("write_operations") / F.col("read_requests")
    ).otherwise(0)
)

In [0]:
# Columnas con NaN por división por cero o nulls
fill_zero = ["max_payload_bytes", "min_payload_bytes", "write_read_ratio"]
df_fe = df_fe.fillna(0, subset=fill_zero)

## Features temporales (semana, dia , hora)

In [0]:
from pyspark.sql import functions as F

DELTA_FE_PATH  = "/Volumes/workspace/default/network_data/features_fe/"
DELTA_FE2_PATH = "/Volumes/workspace/default/network_data/features_fe2/"

df = spark.read.format("delta").load(DELTA_FE_PATH)

# ── Extraer componentes temporales
df = df \
    .withColumn("_hora",       F.hour("window_start")) \
    .withColumn("_dia_semana", F.dayofweek("window_start"))  # 1=Dom...7=Sab

# ── One-hot horas (solo las que tienen señal: 1-11 y 22)
# Las horas 0, 19, 20, 21 siempre son Normal — no aportan discriminación
horas_utiles = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 22]
for h in horas_utiles:
    df = df.withColumn(f"hour_{h:02d}", F.when(F.col("_hora") == h, 1).otherwise(0))

# ── One-hot días (solo los que tienen ataques)
dias_utiles = {2: "mon", 3: "tue", 4: "wed", 5: "thu"}  # dayofweek: 2=Lun...5=Jue
for dow, nombre in dias_utiles.items():
    df = df.withColumn(f"dow_{nombre}", F.when(F.col("_dia_semana") == dow, 1).otherwise(0))

# ── Franja horaria one-hot
df = df \
    .withColumn("franja_manana",
        F.when((F.col("_hora") >= 6) & (F.col("_hora") < 12), 1).otherwise(0)) \
    .withColumn("franja_tarde",
        F.when((F.col("_hora") >= 12) & (F.col("_hora") < 20), 1).otherwise(0)) \
    .withColumn("franja_noche",
        F.when((F.col("_hora") >= 20) | (F.col("_hora") < 6), 1).otherwise(0))

# ── Eliminar columnas auxiliares
df = df.drop("_hora", "_dia_semana")

# ── Verificar
nuevas_temp = [f"hour_{h:02d}" for h in horas_utiles] + \
              [f"dow_{n}" for n in dias_utiles.values()] + \
              ["franja_manana", "franja_tarde", "franja_noche"]

print(f"Features temporales añadidas: {len(nuevas_temp)}")
print(nuevas_temp)

display(
    df.groupBy("label")
      .agg(*[F.round(F.mean(c), 3).alias(c) for c in nuevas_temp])
      .orderBy("label")
)

## GUARDAR

In [0]:
df_fe.repartition(64) \
     .write \
     .format("delta") \
     .mode("overwrite") \
     .option("overwriteSchema", "true") \
     .save(DELTA_FE_PATH)

print(f"✅ Dataset guardado en: {DELTA_FE_PATH}")

# Verificación rápida
df_check = spark.read.format("delta").load(DELTA_FE_PATH)
total = df_check.count()
nulls_label = df_check.filter(F.col("label").isNull()).count()
print(f"Total ventanas : {total:,}")
print(f"NULLs en label : {nulls_label}")
display(df_check.groupBy("label").count().orderBy("label"))